In [1]:
# Resolve paths consistently when launched from the root or notebooks/.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "topic_embeddings").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook from within the derStandard project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Evaluate related-topic closeness

For **A → B**, rank all other topics by cosine similarity to A and check the rank of expected topic B. Exclude only the query topic. Also evaluate **B → A**: similarity is symmetric, but neighbor ranks need not be. Ties follow embedding-file row order. Validation errors are reported and included as non-hits in the summary.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from topic_embeddings.evaluate_related_topics import RelatedEvaluator, summarize_pairs, summarize_related_main

DATA_PATH = Path("results/sgns/topic_embeddings_normalized.csv")
PAIR_PATH = Path("data/benchmarks/topic_related_candidates.csv")
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
DIRECTIONS = {"a_to_b": ("a", "b"), "b_to_a": ("b", "a")}

# Load topic embeddings
topics = pd.read_csv(DATA_PATH, keep_default_na=False, dtype={"Topic_Id": str})
embedding_columns = sorted(c for c in topics.columns if c.startswith("embedding_"))
print(f"Loaded {len(topics):,} topics × {len(embedding_columns)} dimensions.")

# Load related-topic pairs
pairs = pd.read_csv(PAIR_PATH, keep_default_na=False, dtype=str)
if pairs["pair_id"].eq("").any() or pairs["pair_id"].duplicated().any():
    raise ValueError("Pair IDs must be nonempty and unique.")
print(f"Loaded {len(pairs):,} pairs from {PAIR_PATH}.")
print(pairs.groupby("relation").size().rename("Pairs").to_string())

# Initialize the related-topic evaluator
evaluator = RelatedEvaluator(topics, pairs)


Loaded 1,000 topics × 150 dimensions.
Loaded 645 pairs from data/benchmarks/topic_related_candidates.csv.
relation
close_theme            619
taxonomy_relocation     26


## Manually check a pair

Run setup and the function definitions first. Change the ID, neighbor count, or direction below. `evaluate_pair("RP001")` is silent by default and returns `(metrics, neighbors)`; use `verbose=True` to print details. A CSV row can also be passed directly.

In [3]:
PAIR_ID = "RP004"
TOP_K = 11
DIRECTION = "a_to_b"  # Or "b_to_a"

single_metrics, single_neighbors = evaluator.evaluate_pair(
    PAIR_ID, top_k=TOP_K, verbose=True, direction=DIRECTION
)

RP004 | Relation: taxonomy_relocation | Direction: a_to_b
Comments: automobil was replaced by mobilitaet in 2022
Query: 219 | lifestyle/automobil/tretlager
Expected: 675 | lifestyle/mobilitaet/tretlager
 Rank Topic_Id                             Topic_path  Cosine_similarity  Expected_answer
    1      675         lifestyle/mobilitaet/tretlager           0.678890             True
    2      512          lifestyle/automobil/radkasten           0.597438            False
    3      461         panorama/oesterreich/radfahren           0.593463            False
    4      167      lifestyle/automobil/mitreden-auto           0.566007            False
    5      365            lifestyle/automobil/fahrrad           0.562606            False
    6      374               sport/mehrsport/radsport           0.527500            False
    7      865           lifestyle/mobilitaet/fahrrad           0.508800            False
    8      446 lifestyle/reisen/urlaub-in-oesterreich           0.504210     

## Evaluate all pairs and summarize

`all_results` has two rows per pair, one for each direction, including expected ranks, cosine similarities, hit flags, and errors. Inspect top-5 misses with `all_results.loc[~all_results["top5"]]`, or filter by `relation` and `direction`. Batch evaluation is independent of the standalone settings above.

In [4]:
all_results = evaluator.evaluate_all_pairs(pairs)
errors = all_results.loc[all_results["error"].ne(""), ["pair_id", "direction", "error"]]
print(f"Evaluated {len(pairs)} pairs in both directions ({len(all_results)} checks).")
print(f"Successful checks: {len(all_results) - len(errors)}; errors: {len(errors)}.")
if not errors.empty:
    print("\nValidation errors (included as non-hits):")
    print(errors.to_string(index=False))

summary = summarize_pairs(all_results)
print("\nExpected related topics found in the top results:")
print(summary.to_string(index=False, float_format=lambda value: f"{value:.2f}"))

Evaluated 645 pairs in both directions (1290 checks).
Successful checks: 1290; errors: 0.

Expected related topics found in the top results:
           Relation Direction Cutoff  Matches  Total pairs  Hit rate (%)
        close_theme    a_to_b  Top 5      237          619         38.29
        close_theme    a_to_b  Top 3      172          619         27.79
        close_theme    a_to_b  Top 1       73          619         11.79
        close_theme    b_to_a  Top 5      293          619         47.33
        close_theme    b_to_a  Top 3      244          619         39.42
        close_theme    b_to_a  Top 1      122          619         19.71
taxonomy_relocation    a_to_b  Top 5       10           26         38.46
taxonomy_relocation    a_to_b  Top 3        9           26         34.62
taxonomy_relocation    a_to_b  Top 1        6           26         23.08
taxonomy_relocation    b_to_a  Top 5       14           26         53.85
taxonomy_relocation    b_to_a  Top 3       11           

## Main metrics

For each relation, report `(a_to_b top-5 hit rate + b_to_a top-5 hit rate) / 2`, as a percentage.

In [ ]:
main_metrics = summarize_related_main(all_results)
print(main_metrics.to_string(index=False, float_format=lambda value: f"{value:.2f}"))

            Relation  Average top-5 hit rate (%)
        Close themes                       42.81
Taxonomy relocations                       46.15
